In [1]:
import os
import json
import re
from collections import Counter

In [2]:
os.chdir("..")

In [3]:
!pwd

/Users/evseykirichkov/Dev/shikimori_scout


In [4]:
input_file = "output/anime_full.json"
output_file = "output/anime_clean.json"

ALLOWED_KINDS = {"tv", "movie", "ova", "ona", "special", "tv_special"}
EXCLUDED_RATINGS = {"rx", "r_plus"}

In [5]:
def clean_description(text: str) -> str:
    """Убирает BB-теги из описания, оставляя чистый текст."""
    if not text:
        return ""
    text = re.sub(r'\[character=\d+\](.*?)\[/character\]', r'\1', text)
    text = re.sub(r'\[anime=\d+\](.*?)\[/anime\]', r'\1', text)
    text = re.sub(r'\[manga=\d+\](.*?)\[/manga\]', r'\1', text)
    text = re.sub(r'\[person=\d+\](.*?)\[/person\]', r'\1', text)
    text = re.sub(r'\[i\](.*?)\[/i\]', r'\1', text, flags=re.DOTALL)
    text = re.sub(r'\[b\](.*?)\[/b\]', r'\1', text, flags=re.DOTALL)
    text = re.sub(r'\[\[.*?\]\]', '', text)
    text = re.sub(r'\[[^\[\]]*[\u3000-\u9fff\uff00-\uffef][^\[\]]*\]', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\[|\]', '', text)
    return text.strip()

In [6]:
def normalize(anime: dict) -> dict:
    """Преобразует сырую запись из JSON в плоскую нормализованную структуру."""
    genres, themes, demographics = [], [], []
    for g in anime.get("genres") or []:
        name = g.get("russian") or g.get("name", "")
        if g.get("kind") == "genre":
            genres.append(name)
        elif g.get("kind") == "theme":
            themes.append(name)
        elif g.get("kind") == "demographic":
            demographics.append(name)

    related = [
        {"id": r["anime"]["id"],
         "name": r["anime"].get("russian") or r["anime"].get("name", ""),
         "relation": r.get("relationText", "")}
        for r in anime.get("related") or [] if r.get("anime")
    ]
    
    main_characters = [
        cr["character"].get("russian") or cr["character"].get("name", "")
        for cr in anime.get("characterRoles") or [] if cr.get("character")
    ]
    
    persons = [
        {"name": p["person"].get("russian") or p["person"].get("name", ""), "roles": p.get("rolesRu") or p.get("rolesEn") or []}
        for p in anime.get("personRoles") or [] if p.get("person")
    ]
    
    year = (anime.get("airedOn") or {}).get("year")
    trailer_url = next((v["url"] for v in (anime.get("videos") or []) if v.get("url")), None)

    return {
    "id": anime["id"],
    "mal_id": anime.get("malId"),
    "name": anime.get("name", ""),
    "russian": anime.get("russian", ""),
    "synonyms": anime.get("synonyms") or [],
    "kind": anime.get("kind"),
    "episodes": anime.get("episodes", 0),
    "duration": anime.get("duration", 0),
    "status": anime.get("status", ""),
    "season": anime.get("season"),
    "year": year,
    "score": anime.get("score", 0.0),
    "rating": anime.get("rating", "none"),
    "description": clean_description(anime.get("description") or ""),
    "url": anime.get("url", ""),
    "franchise": anime.get("franchise"),
    "poster_url": (anime.get("poster") or {}).get("originalUrl"),
    "genres": genres,
    "themes": themes,
    "demographics": demographics,
    "studios": [s["name"] for s in anime.get("studios") or [] if s.get("name")],
    "related": related,
    "main_characters": main_characters,
    "persons": persons,
    "trailer_url": trailer_url,
    }

In [7]:
def should_keep(a: dict) -> bool:
    return (
        a.get("kind") in ALLOWED_KINDS and
        a.get("score", 0) > 0 and
        bool(a.get("description")) and
        a.get("status") != "anons" and
        a.get("rating") not in EXCLUDED_RATINGS
    )

In [8]:
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

filtered = [normalize(a) for a in data if should_keep(normalize(a))]

print(f"До {len(data)}, после {len(filtered)}")
print(Counter(a["kind"] for a in filtered))

До 29743, после 6326
Counter({'tv': 3087, 'movie': 1030, 'ova': 691, 'special': 633, 'ona': 572, 'tv_special': 313})


In [9]:
filtered[0]

{'id': '52991',
 'mal_id': '52991',
 'name': 'Sousou no Frieren',
 'russian': 'Провожающая в последний путь Фрирен',
 'synonyms': ['Фрирен, провожающая в последний путь',
  'Frieren at the Funeral'],
 'kind': 'tv',
 'episodes': 28,
 'duration': 24,
 'status': 'released',
 'season': 'fall_2023',
 'year': 2023,
 'score': 9.29,
 'rating': 'pg_13',
 'description': 'Одержав победу над Королём демонов, отряд героя Химмеля вернулся домой. Приключение, растянувшееся на десятилетие, подошло к завершению. Волшебница-эльф Фрирен и её отважные товарищи принесли людям мир и разошлись в разные стороны, чтобы спокойно прожить остаток жизни. Однако не всех членов отряда ждёт одинаковая участь. Для эльфов время течёт иначе, поэтому Фрирен вынужденно становится свидетелем того, как её спутники один за другим постепенно уходят из жизни. Девушка осознала, что годы, проведённые в отряде героя, пронеслись в один миг, как падающая звезда в бескрайнем космосе её жизни, и столкнулась с сожалениями об упущенных

In [10]:
total = len(data)
normalized = [normalize(a) for a in data]
no_kind = sum(1 for a in normalized if a.get("kind") not in ALLOWED_KINDS)
no_score = sum(1 for a in normalized if a.get("kind") in ALLOWED_KINDS and a.get("score", 0) == 0)
no_desc = sum(1 for a in normalized if a.get("kind") in ALLOWED_KINDS and a.get("score", 0) > 0 and not a.get("description"))
anons = sum(1 for a in normalized if a.get("kind") in ALLOWED_KINDS and a.get("score", 0) > 0 and a.get("description") and a.get("status") == "anons")

print(f"Всего: {len(data)}")
print(f"Отфильтровано по kind (music/pv/cm/null): {no_kind}")
print(f"Отфильтровано по score == 0: {no_score}")
print(f"Отфильтровано по description == null: {no_desc}")
print(f"Отфильтровано по status == anons: {anons}")
print(f"Осталось: {total - no_kind - no_score - no_desc - anons}")

Всего: 29743
Отфильтровано по kind (music/pv/cm/null): 5140
Отфильтровано по score == 0: 8275
Отфильтровано по description == null: 7917
Отфильтровано по status == anons: 0
Осталось: 8411


In [11]:
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=2)

In [12]:
def build_document(a: dict) -> str:
    """Собирает текстовый документ для embedding (только смысловые поля)."""
    parts = []

    parts.append(f"Название: {a['russian']}")
    if a["genres"]:
        parts.append(f"Жанры: {', '.join(a['genres'])}")
    if a["themes"]:
        parts.append(f"Темы: {', '.join(a['themes'])}")
    if a["demographics"]:
        parts.append(f"Аудитория: {', '.join(a['demographics'])}")
    if a["studios"]:
        parts.append(f"Студия: {', '.join(a['studios'])}")
    if a["main_characters"]:
        parts.append(f"Главные герои: {', '.join(a['main_characters'])}")
    if a["description"]:
        parts.append(f"\nОписание:\n{a['description']}")

    return "\n".join(parts)

In [13]:
print(build_document(filtered[1]))

Название: Провожающая в последний путь Фрирен 2
Жанры: Приключения, Драма, Фэнтези
Аудитория: Сёнен
Студия: Madhouse
Главные герои: Ферн, Фрирен, Штарк

Описание:
Эльфийская волшебница Фрирен, в прошлом одолевшая Короля демонов вместе со своими героическими товарищами Химмелем, Хайтером и Айзеном, продолжает странствовать по миру. Пытаясь лучше понять чувства людей и исполнить последние желания ушедших друзей, Фрирен отправляется в путешествие со своей ученицей Ферн и воином Штарком. Её путь лежит на север, где леденящая опасность скрывается не только в ненастной погоде, но и в намерениях местных обитателей. По дороге к месту, которое, по преданию, является раем на земле, Фрирен и компании предстоит как встретиться с новыми друзьями, так и столкнуться лицом к лицу с могущественным злом, затаившимся в чаще непролазного леса.


In [14]:
import tiktoken

enc = tiktoken.encoding_for_model("text-embedding-3-small")

with open("output/anime_clean.json", "r", encoding="utf-8") as f:
    filtered = json.load(f)

total_tokens = sum(len(enc.encode(build_document(a))) for a in filtered)

print(total_tokens, total_tokens // len(filtered), total_tokens / 1000000 * 3)

2248563 355 6.745689


In [ ]:
import os
import json
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from langchain_openai import OpenAIEmbeddings
from tqdm import tqdm

load_dotenv()

embed_model = "google/gemini-embedding-001"
collection_name = "anime_storage"

embeddings_model = OpenAIEmbeddings(
    api_key=os.environ["VSELLM_API_KEY"],
    model=embed_model,
    base_url=os.environ["VSELLM_BASE_URL"],
)

client = QdrantClient(
    host=os.environ.get("QDRANT_HOST", "localhost"),
    port=int(os.environ.get("QDRANT_PORT", 6333)),
)

In [18]:
client.delete_collection(collection_name)

True

In [19]:
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )

In [20]:
def upload(filtered: list):
    """Векторизует документы батчами и загружает в Qdrant."""
    batch_size = 64
    for i in tqdm(range(0, len(filtered), batch_size)):
        batch = filtered[i:i+batch_size]
        texts = [build_document(a) for a in batch]
        vectors = embeddings_model.embed_documents(texts)
        points = [
            PointStruct(
                id=int(a["id"]),
                vector=vector,
                payload={
                    "russian": a["russian"],
                    "name": a["name"],
                    "description": a["description"],
                    "url": a["url"],
                    "poster_url": a["poster_url"],
                    "score": a["score"],
                    "kind": a["kind"],
                    "episodes": a["episodes"],
                    "duration": a["duration"],
                    "year": a["year"],
                    "status": a["status"],
                    "rating": a["rating"],
                    "genres": a["genres"],
                    "themes": a["themes"],
                    "demographics": a["demographics"],
                    "studios": a["studios"],
                    "franchise": a["franchise"],
                    "trailer_url": a["trailer_url"],
                    "related": a["related"],
                    "persons": a["persons"],
                    "main_characters": a["main_characters"],
                }
            )
            for a, vector in zip(batch, vectors)
        ]
        client.upsert(collection_name=collection_name, points=points)

In [21]:
upload(filtered)

100%|██████████| 132/132 [21:19<00:00,  9.69s/it]


In [32]:
def search(query: str, top_k: int = 3):
    """Ищем аниме по смысловому запросу."""
    vector = embeddings_model.embed_query(query)
    results = client.query_points(
        collection_name=collection_name,
        query=vector,
        limit=top_k
    ).points
    for r in results:
        p = r.payload
        print(f"{p['russian']} | {r.score:.3f} \n{p['description']}")

In [33]:
search("хочу мрачное психологическое аниме")

Мыло для рук | 0.725 
Атмосферное и мрачное видение подросткового возраста и семейной жизни.
Тёмная книга мира | 0.711 
В этом аниме собраны истории странного и чуждого нашему миру, включая истории об НЛО, проклятиях, древних цивилизациях, сверхъестественных силах, опытах общения с духами, необъяснимых происшествиях, а также истории о существовании разных измерений и городские легенды.

Стилистика сериала стремится вызвать ретро-атмосферу печатных сюжетов, которые были популярны в 60-х и 70-х годах XX века.
Цифровой сок | 0.695 
Серия коротких анимационных фильмов, которые показывают различные миры и их героев. Эти эпизоды созданы, чтобы погрузить зрителя в психологический мир фантазий и мистики.


In [16]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

llm_fast = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ["VSELLM_API_KEY"],
    base_url="https://api.vsellm.ru/v1",
)

In [17]:
import sys
sys.path.insert(0, "src")


from tools.vector_db_search import AnimeFilters, build_qdrant_filter
from model.prompt import PROMPT_EXTRACT_FILTERS


def extract_filters(query: str) -> AnimeFilters:
    llm = llm_fast.with_structured_output(AnimeFilters)
    messages = PROMPT_EXTRACT_FILTERS.format_messages(query=query)
    try:
        return llm.invoke(messages)
    except Exception as e:
        print(f"Ошибка: {e}")
        return AnimeFilters()


def validate(query: str):
    filters = extract_filters(query)
    qdrant_filter = build_qdrant_filter(filters)

    print(f"Запрос:  {query}")
    print(f"Фильтры: {filters.model_dump(exclude_none=True)}")
    print(f"Qdrant:  {qdrant_filter}")

In [18]:
validate("хочу что-нибудь про природу, острова и викингов с экшеном и драмой")

Запрос:  хочу что-нибудь про природу, острова и викингов с экшеном и драмой
Фильтры: {'genres_include': ['Экшен', 'Драма']}
Qdrant:  should=None min_should=None must=[FieldCondition(key='genres', match=MatchAny(any=['Экшен', 'Драма']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=[FieldCondition(key='rating', match=MatchAny(any=['rx', 'r_plus']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)]


/opt/anaconda3/envs/shiki_env/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AnimeFilters(genres_inclu...None, duration_max=None), input_type=AnimeFilters])
  return self.__pydantic_serializer__.to_python(


In [91]:
!pwd

/Users/evseykirichkov/Dev/shikimori_scout


In [92]:
anime_clean = json.load(open("output/anime_clean.json"))

In [96]:
import json
import sqlite3
from pathlib import Path

In [97]:
DB_PATH = Path("data/anime.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Основная таблица
cur.execute("DROP TABLE IF EXISTS anime_genre")
cur.execute("DROP TABLE IF EXISTS anime_theme")
cur.execute("DROP TABLE IF EXISTS anime_demographic")
cur.execute("DROP TABLE IF EXISTS anime_studio")
cur.execute("DROP TABLE IF EXISTS genre")
cur.execute("DROP TABLE IF EXISTS theme")
cur.execute("DROP TABLE IF EXISTS demographic")
cur.execute("DROP TABLE IF EXISTS studio")
cur.execute("DROP TABLE IF EXISTS anime")

cur.execute("""
    CREATE TABLE anime (
        id          INTEGER PRIMARY KEY,
        mal_id      INTEGER,
        name        TEXT,
        russian     TEXT,
        kind        TEXT,
        episodes    INTEGER,
        duration    INTEGER,
        status      TEXT,
        year        INTEGER,
        score       REAL,
        rating      TEXT,
        description TEXT,
        url         TEXT,
        poster_url  TEXT,
        trailer_url TEXT,
        franchise   TEXT
    )
""")

# Справочники
for table in ("genre", "theme", "demographic", "studio"):
    cur.execute(f"CREATE TABLE {table} (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE)")

# Связи many-to-many
for table in ("genre", "theme", "demographic", "studio"):
    cur.execute(f"""
        CREATE TABLE anime_{table} (
            anime_id INTEGER REFERENCES anime(id),
            {table}_id INTEGER REFERENCES {table}(id),
            PRIMARY KEY (anime_id, {table}_id)
        )
    """)

# Вставка данных
def get_or_create(table, name):
    cur.execute(f"INSERT OR IGNORE INTO {table} (name) VALUES (?)", (name,))
    cur.execute(f"SELECT id FROM {table} WHERE name = ?", (name,))
    return cur.fetchone()[0]

for a in anime_clean:
    cur.execute(
        "INSERT OR REPLACE INTO anime VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
        (
            int(a["id"]), int(a["mal_id"]) if a.get("mal_id") else None,
            a.get("name"), a.get("russian"), a.get("kind"),
            a.get("episodes", 0), a.get("duration", 0),
            a.get("status"), a.get("year"), a.get("score", 0.0),
            a.get("rating"), a.get("description"), a.get("url"),
            a.get("poster_url"), a.get("trailer_url"), a.get("franchise"),
        ),
    )
    aid = int(a["id"])
    for g in a.get("genres") or []:
        gid = get_or_create("genre", g)
        cur.execute("INSERT OR IGNORE INTO anime_genre VALUES (?,?)", (aid, gid))
    for t in a.get("themes") or []:
        tid = get_or_create("theme", t)
        cur.execute("INSERT OR IGNORE INTO anime_theme VALUES (?,?)", (aid, tid))
    for d in a.get("demographics") or []:
        did = get_or_create("demographic", d)
        cur.execute("INSERT OR IGNORE INTO anime_demographic VALUES (?,?)", (aid, did))
    for s in a.get("studios") or []:
        sid = get_or_create("studio", s)
        cur.execute("INSERT OR IGNORE INTO anime_studio VALUES (?,?)", (aid, sid))

cur.execute("CREATE INDEX IF NOT EXISTS idx_score  ON anime(score DESC)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_year   ON anime(year)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_kind   ON anime(kind)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_rating ON anime(rating)")

In [99]:
conn.commit()
conn.close()
print(f"База готова: {DB_PATH} ({len(rows)} записей)")

База готова: data/anime.db (8411 записей)


In [ ]:
conn = sqlite3.connect("data/anime.db")
cur = conn.cursor()

cur.execute("SELECT id, russian, score, genres, themes, studios FROM anime WHERE russian = 'Провожающая в последний путь Фрирен'")
row = cur.fetchone()
print(row)

('52991', 'Провожающая в последний путь Фрирен', 9.29, '["Приключения", "Драма", "Фэнтези"]', '[]', '["Madhouse"]')


In [101]:
cur.execute("SELECT russian, score FROM anime ORDER BY score DESC LIMIT 5")
for r in cur.fetchall():
    print(r)

('Провожающая в последний путь Фрирен', 9.29)
('Провожающая в последний путь Фрирен 2', 9.17)
('Стальной алхимик: Братство', 9.1)
('Человек-бензопила: Резе', 9.09)
('Врата Штейна', 9.07)


In [102]:
conn.close()